# Giai Đoạn 0: Demo Mạng Tích Chập Lượng Tử (Quanvolutional Neural Network)

**Đề tài Luận văn:** Nghiên cứu và ứng dụng lớp tích chập lượng tử (Quanvolution) cho phân loại ảnh y tế (MedMNIST)

---

## Mục tiêu:
1. **Hiểu cơ chế hoạt động:** Xây dựng mạch lượng tử 4-qubit trượt qua các vùng ảnh cục bộ (patch $2 \times 2$).
2. **Quy trình mã hóa & đo lường:** Mã hóa pixel thành góc quay $R_Y(\pi \cdot \text{pixel})$, áp dụng lớp vướng víu ngẫu nhiên (`RandomLayers`) và đo kỳ vọng Pauli-Z $\langle Z \rangle$.
3. **Trực quan hóa:** Hiển thị và so sánh ảnh gốc ($28 \times 28$) với 4 kênh đặc trưng lượng tử ($14 \times 14$).
4. **Phân loại thử nghiệm:** Nối các kênh đặc trưng lượng tử vào một bộ phân loại cổ điển đơn giản (Linear Classifier) trên PyTorch.

### 1. Cài đặt & Khai báo các Thư viện Cần thiết

In [ ]:
import random
import pennylane as qml
from pennylane import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# ==========================================
# Cố định Seed đảm bảo Tính Tái lập (Reproducibility)
# ==========================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"PennyLane version: {qml.__version__}")

### 2. Chuẩn bị Dữ liệu (MNIST Subset cho Demo)

In [ ]:
n_train = 50   # Số lượng mẫu huấn luyện demo
n_test = 30    # Số lượng mẫu kiểm thử demo

transform = transforms.Compose([
    transforms.ToTensor(), # Chuẩn hóa pixel về đoạn [0, 1]
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Lấy subset nhỏ để demo diễn ra nhanh chóng trên CPU
train_subset = Subset(train_dataset, range(n_train))
test_subset = Subset(test_dataset, range(n_test))

train_loader = DataLoader(train_subset, batch_size=1, shuffle=False)
test_loader = DataLoader(test_subset, batch_size=1, shuffle=False)

print(f"Đã tải {len(train_subset)} mẫu train và {len(test_subset)} mẫu test.")

### 3. Thiết kế Mạch Lượng Tử (Quanvolution Layer / Quantum Kernel)

Mạch gồm **4 qubits** ứng với 4 pixel của kernel $2 \times 2$:
1. **Mã hóa góc (Angle Encoding):** Dùng cổng xoay $R_Y(\pi \cdot \text{pixel})$.
2. **Biến đổi lượng tử (Quantum Entanglement):** Sử dụng `qml.RandomLayers` cố định.
3. **Đo lường (Measurement):** Đo giá trị kỳ vọng Pauli-Z $\langle Z \rangle$ trên từng qubit, trả về vector 4 chiều.

In [ ]:
n_layers = 1 # Số lớp biến đổi ngẫu nhiên
dev = qml.device("default.qubit", wires=4)

# Tham số ngẫu nhiên cố định cho lớp lượng tử
rand_params = np.random.uniform(high=2 * np.pi, size=(n_layers, 4))

@qml.qnode(dev, interface="torch")
def quantum_circuit(phi):
    # 1. Angle Encoding cho 4 pixel
    for j in range(4):
        qml.RY(np.pi * phi[j], wires=j)

    # 2. Random Entanglement Layer
    qml.RandomLayers(rand_params, wires=list(range(4)))

    # 3. Measurement (Kỳ vọng Pauli-Z)
    return [qml.expval(qml.PauliZ(j)) for j in range(4)]

# In sơ đồ kiến trúc mạch lượng tử
dummy_input = np.array([0.1, 0.2, 0.3, 0.4])
print("Sơ đồ Mạch Lượng tử (Quantum Circuit Diagram):")
print(qml.draw(quantum_circuit)(dummy_input))

### 4. Định nghĩa Hàm Quét Kernel Lượng Tử (`quanv`)

In [ ]:
def quanv(image):
    """
    Áp dụng bộ lọc Quanvolution lên ảnh 2D.
    Kernel: 2x2, Stride: 2 (không overlapping).
    Input: Tensor (1, 28, 28) -> Output: Tensor (4, 14, 14)
    """
    out = torch.zeros((4, 14, 14))
    
    for j in range(0, 28, 2):
        for k in range(0, 28, 2):
            # Trích xuất patch 2x2 gồm 4 pixel lân cận
            patch = [
                image[0, j, k],
                image[0, j, k + 1],
                image[0, j + 1, k],
                image[0, j + 1, k + 1]
            ]
            # Đưa qua mạch lượng tử
            q_results = quantum_circuit(patch)
            
            # Gán kết quả đo vào 4 kênh đặc trưng
            for c in range(4):
                out[c, j // 2, k // 2] = q_results[c]
                
    return out

### 5. Trực Quan Hóa Feature Maps Trích Xuất từ Mạch Lượng Tử

In [ ]:
# Lấy thử 1 ảnh từ tập train
sample_img, sample_label = next(iter(train_loader))
sample_img = sample_img[0] # Tensor shape: (1, 28, 28)

print(f"Shape ảnh gốc: {sample_img.shape}")
q_features = quanv(sample_img)
print(f"Shape sau khi qua Quantum Kernel: {q_features.shape}")

# Vẽ hiển thị trực tiếp trong Notebook
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
axes[0].imshow(sample_img[0].numpy(), cmap='gray')
axes[0].set_title(f"Ảnh Gốc (28x28)\nLabel: {sample_label.item()}", fontsize=11)
axes[0].axis('off')

for i in range(4):
    axes[i+1].imshow(q_features[i].detach().numpy(), cmap='magma')
    axes[i+1].set_title(f"Quantum Channel {i}\n(14x14)", fontsize=11)
    axes[i+1].axis('off')

plt.tight_layout()
plt.show()

### 6. Tiền Xử Lý (Precompute) Đặc Trưng Lượng Tử Cho Toàn Bộ Dữ Liệu

> **Chiến lược Precomputation:** Do mạch lượng tử là cố định (*Random static circuit*), ta tính trước toàn bộ Feature Maps cho các ảnh một lần duy nhất để quá trình huấn luyện mạng cổ điển diễn ra siêu tốc (chỉ mất vài giây).

In [ ]:
print("Đang tiền tính đặc trưng cho Train Set...")
q_train_images = [quanv(img[0]) for img, _ in train_loader]
q_train_images = torch.stack(q_train_images)
train_labels = torch.tensor([label for _, label in train_subset])

print("Đang tiền tính đặc trưng cho Test Set...")
q_test_images = [quanv(img[0]) for img, _ in test_loader]
q_test_images = torch.stack(q_test_images)
test_labels = torch.tensor([label for _, label in test_subset])

print(f"Hoàn tất! Train features shape: {q_train_images.shape}, Test features shape: {q_test_images.shape}")

### 7. Xây Dựng & Huấn Luyện Mạng Phân Loại Cổ Điển (Linear Classifier)

In [ ]:
class SimpleClassifier(nn.Module):
    def __init__(self):
        super(SimpleClassifier, self).__init__()
        # Đầu vào: 4 channels * 14 * 14 = 784
        # Đầu ra: 10 classes (chữ số 0-9)
        self.fc = nn.Linear(4 * 14 * 14, 10)

    def forward(self, x):
        x = x.view(x.shape[0], -1) # Flatten
        return self.fc(x)

model = SimpleClassifier()
optimizer = torch.optim.SGD(model.parameters(), lr=0.02)
criterion = nn.CrossEntropyLoss()
n_epochs = 10

print("Bắt đầu Huấn Luyện:")
for epoch in range(n_epochs):
    model.train()
    total_loss = 0.0
    for i in range(len(q_train_images)):
        optimizer.zero_grad()
        out = model(q_train_images[i].unsqueeze(0))
        loss = criterion(out, train_labels[i].unsqueeze(0))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    # Đánh giá trên tập test
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(len(q_test_images)):
            out = model(q_test_images[i].unsqueeze(0))
            if out.argmax(dim=1) == test_labels[i]:
                correct += 1
    
    acc = correct / len(q_test_images)
    print(f"Epoch {epoch+1:02d}/{n_epochs:02d} | Avg Loss: {total_loss/len(q_train_images):.4f} | Test Accuracy: {acc * 100:.2f}%")